In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import sqlite3
import pandas as pd

DB_PATH = PROJECT_ROOT / "data" / "nifty100.db"

conn = sqlite3.connect(DB_PATH)

In [3]:
companies = pd.read_sql(
    """
    SELECT
        company_id,
        roe_percentage,
        roce_percentage
    FROM companies
    """,
    conn
)

ratios = pd.read_sql(
    """
    SELECT
    company_id,
    year,
    return_on_equity_pct
FROM financial_ratios
    """,
    conn
)

companies.head()

,company_id,roe_percentage,roce_percentage
0,ABB,34.90,46.0
1,ADANIENSOL,8.59,9.0
2,ADANIENT,13.64,11.6
3,ADANIGREEN,14.70,96.5
4,ADANIPORTS,18.10,12.9


In [3]:
latest = (
    ratios.sort_values("year")
          .groupby("company_id")
          .tail(1)
)

validation = latest.merge(
    companies,
    on="company_id",
    how="left"
)

validation.head()

,company_id,year,return_on_equity_pct,return_on_assets_pct,roe_percentage,roce_percentage
0,DRREDDY,2024.0,19.742337,14.383703,21.40,26.50
1,DMART,2024.0,13.562948,11.978084,14.50,19.40
2,TORNTPHARM,2024.0,24.154026,11.416753,24.20,23.20
3,EICHERMOT,2024.0,22.172347,17.309107,24.20,31.10
4,DLF,2024.0,6.908270,4.611556,6.95,5.74


In [4]:
validation["roe_difference"] = (
    validation["return_on_equity_pct"]
    - validation["roe_percentage"]
).abs()

validation["roce_difference"] = (
    validation["return_on_assets_pct"]
    - validation["roce_percentage"]
).abs()

validation.head()

,company_id,year,return_on_equity_pct,return_on_assets_pct,roe_percentage,roce_percentage,roe_difference,roce_difference
0,DRREDDY,2024.0,19.742337,14.383703,21.40,26.50,1.657663,12.116297
1,DMART,2024.0,13.562948,11.978084,14.50,19.40,0.937052,7.421916
2,TORNTPHARM,2024.0,24.154026,11.416753,24.20,23.20,0.045974,11.783247
3,EICHERMOT,2024.0,22.172347,17.309107,24.20,31.10,2.027653,13.790893
4,DLF,2024.0,6.908270,4.611556,6.95,5.74,0.041730,1.128444


In [5]:
roe_issues = validation[
    validation["roe_difference"] > 5
]

roce_issues = validation[
    validation["roce_difference"] > 5
]

print("ROE Issues:", len(roe_issues))
print("ROCE Issues:", len(roce_issues))

ROE Issues: 18
ROCE Issues: 81


In [6]:
OUTPUT = PROJECT_ROOT / "output"

with open(
    OUTPUT / "ratio_edge_cases.log",
    "a"
) as f:

    f.write("\n========== ROE VALIDATION ==========\n")

    for _, row in roe_issues.iterrows():

        f.write(
            f"{row.company_id}: "
            f"Calculated={row.return_on_equity_pct:.2f}, "
            f"Source={row.roe_percentage:.2f}\n"
        )

    f.write("\n========== ROCE VALIDATION ==========\n")

    for _, row in roce_issues.iterrows():

        f.write(
            f"{row.company_id}: "
            f"Calculated={row.return_on_assets_pct:.2f}, "
            f"Source={row.roce_percentage:.2f}\n"
        )

print("Validation log updated.")

Validation log updated.
